# Amazon Search / LTR Project — Week 3 Prompt V2.1
## Canonical Attribute Schema + Fair Normalized Evaluation

Prompt V2 improved intent classification and retrieval, but its attribute score was distorted by an inconsistent schema.

This notebook fixes the evaluation definition **before further prompt tuning**.

### Key design change
`brand`, `category`, `core_product`, and `use_case` are separate structured fields and are **not counted again as attributes**.

The `attributes` list is restricted to one canonical vocabulary:

`color, size, material, gender, quantity, style, shade, model, product_line,
compatibility, finish, scent, feature, benefit, spf, gauge, location, theme,
skin_type, volume, taste, hold, shape, ingredient, target_audience, purpose,
application`

### Experimental discipline
The original 50 manually labeled queries are treated as a **development validation set**.
After Prompt V2.1 is stable, use the remaining fixed Week-2 test queries for the larger run instead of repeatedly optimizing on the same 50.

### Required uploads
- `amazon_week1_v4_processed_data.zip`
- `amazon_week2_baseline_ranking_results.zip`
- `week3_manual_validation_50_normalized.csv`

In [ ]:
!pip -q install pandas numpy pyarrow rank-bm25 scikit-learn tqdm requests matplotlib

In [ ]:
MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"

RANDOM_SEED = 42
DEV_N = 50
TARGET_ATTRIBUTE_F1 = 0.75
TARGET_INTENT_ACCURACY = 0.60

# Run larger holdout queries only after the 50-query development set is satisfactory.
RUN_REMAINING_TEST_QUERIES = False

## 1. Colab-safe Ollama setup

In [ ]:
import os
import time
import shutil
import subprocess
import requests
import platform
from pathlib import Path

def ollama_alive():
    try:
        return requests.get(
            f"{OLLAMA_BASE_URL}/api/tags",
            timeout=2
        ).ok
    except Exception:
        return False

def run_cmd(cmd):
    print("+", cmd)
    r = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

if shutil.which("ollama") is None:
    machine = platform.machine().lower()
    arch = "amd64" if machine in ("x86_64", "amd64") else "arm64"

    run_cmd("apt-get -qq update")
    run_cmd("apt-get -qq install -y zstd curl")

    url = f"https://ollama.com/download/ollama-linux-{arch}.tar.zst"
    run_cmd(
        f"curl -fL --retry 5 --retry-delay 2 "
        f"{url} -o /tmp/ollama.tar.zst"
    )
    run_cmd(
        "tar --use-compress-program=unzstd "
        "-xf /tmp/ollama.tar.zst -C /usr"
    )

if not ollama_alive():
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"

    log = open("/tmp/ollama_v21.log", "w")
    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=log,
        stderr=subprocess.STDOUT,
        env=env
    )

    for _ in range(90):
        if ollama_alive():
            break
        time.sleep(1)

assert ollama_alive(), "Ollama server did not start."
subprocess.run(["ollama", "pull", MODEL], check=True)
print("Ollama ready:", MODEL)

## 2. Upload inputs

In [ ]:
from pathlib import Path
import zipfile

EXPECTED = [
    "amazon_week1_v4_processed_data.zip",
    "amazon_week2_baseline_ranking_results.zip",
    "week3_manual_validation_50_normalized.csv",
]

def in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

missing = [f for f in EXPECTED if not Path(f).exists()]

if missing and in_colab():
    from google.colab import files
    files.upload()

missing = [f for f in EXPECTED if not Path(f).exists()]
if missing:
    raise FileNotFoundError("Missing: " + ", ".join(missing))

WORKDIR = Path("amazon_week3_v21")
INPUT_DIR = WORKDIR / "inputs"
RESULTS_DIR = WORKDIR / "results"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for zip_name in EXPECTED[:2]:
    target = INPUT_DIR / Path(zip_name).stem
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(target)

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from IPython.display import display

W1_DIR = INPUT_DIR / "amazon_week1_v4_processed_data"
W2_DIR = INPUT_DIR / "amazon_week2_baseline_ranking_results"

qpr = pd.read_csv(W1_DIR / "query_product_relevance.csv")
products = pd.read_parquet(W1_DIR / "cleaned_products_10k.parquet")
fixed_test = pd.read_csv(W2_DIR / "fixed_test_queries.csv")
gold = pd.read_csv("week3_manual_validation_50_normalized.csv")

for df in (qpr, fixed_test, gold):
    df["query_id"] = df["query_id"].astype(str)

qpr["product_id"] = qpr["product_id"].astype(str)
products["product_id"] = products["product_id"].astype(str)

query_table = (
    qpr[["query_id", "query"]]
    .drop_duplicates("query_id")
    .merge(fixed_test[["query_id"]], on="query_id", how="inner")
)

order = {qid: i for i, qid in enumerate(fixed_test["query_id"].tolist())}
query_table["_order"] = query_table["query_id"].map(order)
query_table = (
    query_table.sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

dev_ids = gold["query_id"].tolist()
dev = query_table[query_table["query_id"].isin(dev_ids)].copy()
dev["_order"] = dev["query_id"].map({qid: i for i, qid in enumerate(dev_ids)})
dev = dev.sort_values("_order").drop(columns="_order").reset_index(drop=True)

print("Development queries:", len(dev))
print("Fixed test queries total:", len(query_table))

## 3. Prompt V2.1 — canonical schema

Important changes from V2:

- Attribute names are restricted to a fixed enum.
- `brand`, `category`, `product type`, and `use case` must not be duplicated inside `attributes`.
- `shade` should include shade code + shade name together when both belong to the same shade (e.g. `765 medium`).
- `model` is reserved for actual model/SKU/product codes.
- Sizes and decimals must be preserved exactly.
- Product lines should go to `product_line`, not `color`, `brand`, or `material`.

In [ ]:
ATTRIBUTE_NAMES = [
    "color", "size", "material", "gender", "quantity", "style",
    "shade", "model", "product_line", "compatibility", "finish",
    "scent", "feature", "benefit", "spf", "gauge", "location",
    "theme", "skin_type", "volume", "taste", "hold", "shape",
    "ingredient", "target_audience", "purpose", "application"
]

QUERY_SCHEMA = {
    "type": "object",
    "properties": {
        "query": {"type": "string"},
        "intent": {
            "type": "string",
            "enum": [
                "exact_product_search",
                "product_search",
                "category_browse",
                "brand_search",
                "use_case_search",
                "compatibility_search",
                "other"
            ]
        },
        "core_product": {"type": ["string", "null"]},
        "brand": {"type": ["string", "null"]},
        "category": {"type": ["string", "null"]},
        "attributes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "enum": ATTRIBUTE_NAMES
                    },
                    "value": {"type": "string"},
                    "explicit": {"type": "boolean"}
                },
                "required": ["name", "value", "explicit"],
                "additionalProperties": False
            }
        },
        "price_min": {"type": ["number", "null"]},
        "price_max": {"type": ["number", "null"]},
        "use_case": {"type": ["string", "null"]},
        "implicit_requirements": {
            "type": "array",
            "items": {"type": "string"}
        },
        "rewritten_query": {"type": "string"}
    },
    "required": [
        "query", "intent", "core_product", "brand", "category",
        "attributes", "price_min", "price_max", "use_case",
        "implicit_requirements", "rewritten_query"
    ],
    "additionalProperties": False
}

PROMPT_V21 = """
You are a conservative e-commerce search query-understanding system.

Return exactly one JSON object using the supplied schema.

INTENT
- exact_product_search: unique named product, product line, model, shade/code, or specific variant.
- brand_search: brand + broad product/category, without a unique item/model.
- use_case_search: need/problem/outcome but no concrete product.
- category_browse: broad category browsing.
- product_search: generic named product type with optional attributes.
- compatibility_search: compatibility is central.
- other: review fragment, subjective comment, incomplete/ambiguous text, or no confident product/category.

SEPARATE FIELDS
The following concepts have their own top-level fields and MUST NOT be repeated inside attributes:
- brand
- category
- core product / product type
- use case
- price

CANONICAL ATTRIBUTES
Use only the allowed attribute names from the schema.

IMPORTANT ATTRIBUTE RULES
- product_line: named family/line such as "Nice 'n Easy" or "Yellow".
- shade: combine shade code and shade name when they form one shade, e.g. "765 medium".
- model: actual model/SKU/product code such as "PMS061", "Surf-350", "1090".
- size: preserve the full decimal and unit, e.g. "33.8oz", "18.75oz", "3g".
- volume: product strength/volume designation when it is not physical package size, e.g. "20 volume".
- feature: generic property such as "organic", "wildcrafted", "long lasting", "permanent", "waterproof".
- ingredient: named ingredient such as "mint" or "myrrh".
- benefit: desired effect such as "radiance" or "even tan".

DO NOT
- create attribute names outside the schema.
- use product_type, brand, category, or use_case as an attribute.
- split decimals.
- turn shade/model codes into quantity or size.
- infer color from a product-line/brand word.
- hallucinate a product category from a review fragment.

REWRITE
Preserve every explicit brand, product line, model, shade/code, decimal, size/unit, and quantity.
Remove conversational filler only.
Never replace a specific product with a more generic or different product.

Output JSON only.
""".strip()

In [ ]:
import requests
import time
from tqdm.auto import tqdm

def call_v21(query, retries=3):
    messages = [
        {"role": "system", "content": PROMPT_V21},
        {"role": "user", "content": f"Query: {query}"}
    ]

    last_error = None

    for attempt in range(1, retries + 1):
        start = time.perf_counter()
        payload = {
            "model": MODEL,
            "messages": messages,
            "stream": False,
            "format": QUERY_SCHEMA,
            "options": {
                "temperature": 0,
                "seed": RANDOM_SEED,
                "num_ctx": 4096
            }
        }

        try:
            r = requests.post(
                f"{OLLAMA_BASE_URL}/api/chat",
                json=payload,
                timeout=180
            )

            if r.status_code >= 400:
                payload["format"] = "json"
                r = requests.post(
                    f"{OLLAMA_BASE_URL}/api/chat",
                    json=payload,
                    timeout=180
                )

            r.raise_for_status()
            obj = json.loads(r.json()["message"]["content"])
            obj["query"] = query

            return obj, time.perf_counter() - start, None

        except Exception as e:
            last_error = repr(e)
            time.sleep(min(2 * attempt, 6))

    return None, np.nan, last_error

## 4. Run V2.1 on the 50-query development set

In [ ]:
rows = []

for r in tqdm(
    dev.itertuples(index=False),
    total=len(dev),
    desc="Prompt V2.1 dev validation"
):
    obj, latency, error = call_v21(r.query)
    obj = obj or {}

    rows.append({
        "query_id": r.query_id,
        "query": r.query,
        "v21_intent": obj.get("intent"),
        "v21_core_product": obj.get("core_product"),
        "v21_brand": obj.get("brand"),
        "v21_category": obj.get("category"),
        "v21_attributes_json": json.dumps(
            obj.get("attributes", []),
            ensure_ascii=False
        ),
        "v21_price_min": obj.get("price_min"),
        "v21_price_max": obj.get("price_max"),
        "v21_use_case": obj.get("use_case"),
        "v21_implicit_requirements_json": json.dumps(
            obj.get("implicit_requirements", []),
            ensure_ascii=False
        ),
        "v21_rewritten_query": obj.get("rewritten_query"),
        "v21_latency_sec": latency,
        "v21_error": error
    })

v21 = pd.DataFrame(rows)
v21.to_csv(
    RESULTS_DIR / "week3_prompt_v21_outputs_50.csv",
    index=False
)

display(v21.head(10))

## 5. Fair normalized evaluator

This evaluator compares only canonical attribute pairs.

It does **not** count brand/category/product-type/use-case twice.

In [ ]:
def norm_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower().replace("’", "'").replace("–", "-")
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"(\d)\s+(oz|g|ml|lb|lbs)\b", r"\1\2", s)
    return s

def json_list(x):
    try:
        y = json.loads(x) if isinstance(x, str) and x.strip() else []
        return y if isinstance(y, list) else []
    except Exception:
        return []

def pair_set(x):
    out = set()
    for item in json_list(x):
        if not isinstance(item, dict):
            continue
        name = norm_text(item.get("name"))
        value = norm_text(item.get("value"))
        if name and value:
            out.add((name, value))
    return out

merged = gold.merge(
    v21,
    on=["query_id", "query"],
    how="left"
)

def exact_acc(pred, truth):
    return np.mean([
        norm_text(p) == norm_text(t)
        for p, t in zip(merged[pred], merged[truth])
    ])

tp = fp = fn = 0

for _, r in merged.iterrows():
    pred = pair_set(r["v21_attributes_json"])
    truth = pair_set(r["true_attributes_normalized_json"])

    tp += len(pred & truth)
    fp += len(pred - truth)
    fn += len(truth - pred)

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall
    else 0.0
)

metrics = pd.DataFrame({
    "metric": [
        "intent_accuracy",
        "core_product_accuracy",
        "brand_accuracy",
        "category_accuracy",
        "attribute_precision_normalized",
        "attribute_recall_normalized",
        "attribute_f1_normalized"
    ],
    "value": [
        exact_acc("v21_intent", "true_intent"),
        exact_acc("v21_core_product", "true_core_product"),
        exact_acc("v21_brand", "true_brand"),
        exact_acc("v21_category", "true_category"),
        precision,
        recall,
        f1
    ]
})

metrics.to_csv(
    RESULTS_DIR / "week3_prompt_v21_manual_metrics.csv",
    index=False
)

display(metrics)

## 6. Error analysis

In [ ]:
def attribute_diff(row):
    truth = pair_set(row["true_attributes_normalized_json"])
    pred = pair_set(row["v21_attributes_json"])

    return pd.Series({
        "missing_attributes": json.dumps(
            sorted(list(truth - pred)),
            ensure_ascii=False
        ),
        "extra_attributes": json.dumps(
            sorted(list(pred - truth)),
            ensure_ascii=False
        )
    })

diffs = merged.apply(attribute_diff, axis=1)

errors = pd.concat(
    [
        merged[
            [
                "query_id", "query",
                "true_intent", "v21_intent",
                "true_core_product", "v21_core_product",
                "true_attributes_normalized_json",
                "v21_attributes_json",
                "v21_rewritten_query",
                "notes"
            ]
        ],
        diffs
    ],
    axis=1
)

errors.to_csv(
    RESULTS_DIR / "week3_prompt_v21_error_analysis.csv",
    index=False
)

display(
    errors[
        (errors["missing_attributes"] != "[]")
        | (errors["extra_attributes"] != "[]")
    ].head(25)
)

## 7. BM25 retrieval validation for V2.1

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score

def tokenize(text):
    if pd.isna(text):
        return []
    return re.findall(r"[a-z0-9]+", str(text).lower())

products["title"] = products["title"].fillna("").astype(str)
products["description"] = products["description"].fillna("").astype(str)

documents = (
    products["title"].str.strip()
    + " "
    + products["description"].str.strip()
).str.strip()

bm25 = BM25Okapi(
    [tokenize(x) for x in tqdm(documents, desc="Tokenizing catalog")]
)

product_ids = products["product_id"].tolist()
pid_to_idx = {pid: i for i, pid in enumerate(product_ids)}

def rr(labels):
    for i, x in enumerate(labels, start=1):
        if x >= 1:
            return 1 / i
    return 0.0

def recall_at(retrieved, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return np.nan
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def bm25_scores(q):
    return np.asarray(bm25.get_scores(tokenize(q)))

def eval_version(qid, qtext, version):
    scores = bm25_scores(qtext)

    judged = qpr[qpr["query_id"] == str(qid)].copy()
    judged["idx"] = judged["product_id"].map(pid_to_idx)
    judged = judged[judged["idx"].notna()].copy()
    judged["idx"] = judged["idx"].astype(int)
    judged["score"] = scores[judged["idx"].values]

    ranked = judged.sort_values("score", ascending=False)
    labels = ranked["relevance_label"].astype(float).tolist()

    ndcg = float(
        ndcg_score(
            judged["relevance_label"].astype(float).to_numpy().reshape(1, -1),
            judged["score"].astype(float).to_numpy().reshape(1, -1),
            k=10
        )
    )

    top_idx = np.argpartition(scores, -50)[-50:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    retrieved = [product_ids[i] for i in top_idx]

    strong = set(
        judged.loc[
            judged["relevance_label"] >= 2,
            "product_id"
        ].astype(str)
    )

    return {
        "query_id": str(qid),
        "version": version,
        "ndcg@10": ndcg,
        "mrr": rr(labels),
        "recall@10": recall_at(retrieved, strong, 10),
        "recall@50": recall_at(retrieved, strong, 50)
    }

eval_rows = []

for r in tqdm(
    merged.itertuples(index=False),
    total=len(merged),
    desc="V2.1 retrieval validation"
):
    eval_rows.append(eval_version(r.query_id, r.query, "original"))
    eval_rows.append(
        eval_version(
            r.query_id,
            r.v21_rewritten_query,
            "v21_rewrite"
        )
    )

retrieval = pd.DataFrame(eval_rows)

retrieval_summary = (
    retrieval.groupby("version")[
        ["ndcg@10", "mrr", "recall@10", "recall@50"]
    ]
    .mean()
    .reset_index()
)

retrieval.to_csv(
    RESULTS_DIR / "week3_prompt_v21_retrieval_per_query.csv",
    index=False
)
retrieval_summary.to_csv(
    RESULTS_DIR / "week3_prompt_v21_retrieval_summary.csv",
    index=False
)

display(retrieval_summary)

## 8. Development gate

Recommended gate before using the remaining fixed test queries:

- normalized Attribute F1 ≥ 0.75
- Intent Accuracy ≥ 0.60
- rewritten Recall@50 no worse than 98% of original Recall@50

In [ ]:
intent_acc = float(
    metrics.loc[
        metrics["metric"] == "intent_accuracy",
        "value"
    ].iloc[0]
)

attr_f1 = float(
    metrics.loc[
        metrics["metric"] == "attribute_f1_normalized",
        "value"
    ].iloc[0]
)

original_r50 = float(
    retrieval_summary.loc[
        retrieval_summary["version"] == "original",
        "recall@50"
    ].iloc[0]
)

v21_r50 = float(
    retrieval_summary.loc[
        retrieval_summary["version"] == "v21_rewrite",
        "recall@50"
    ].iloc[0]
)

gate = (
    attr_f1 >= TARGET_ATTRIBUTE_F1
    and intent_acc >= TARGET_INTENT_ACCURACY
    and v21_r50 >= original_r50 * 0.98
)

print("Intent Accuracy:", round(intent_acc, 4))
print("Normalized Attribute F1:", round(attr_f1, 4))
print("Original Recall@50:", round(original_r50, 4))
print("V2.1 Recall@50:", round(v21_r50, 4))
print()
print(
    "DEVELOPMENT GATE:",
    "PASS" if gate else "NOT YET PASSED"
)

## 9. Optional larger run on the remaining fixed test queries

This intentionally excludes the 50 development queries to reduce prompt-tuning leakage.

In [ ]:
if RUN_REMAINING_TEST_QUERIES:
    holdout = query_table[
        ~query_table["query_id"].isin(dev_ids)
    ].copy()

    holdout_rows = []

    for r in tqdm(
        holdout.itertuples(index=False),
        total=len(holdout),
        desc="V2.1 remaining fixed-test queries"
    ):
        obj, latency, error = call_v21(r.query)
        obj = obj or {}

        holdout_rows.append({
            "query_id": r.query_id,
            "query": r.query,
            "intent": obj.get("intent"),
            "core_product": obj.get("core_product"),
            "brand": obj.get("brand"),
            "category": obj.get("category"),
            "attributes_json": json.dumps(
                obj.get("attributes", []),
                ensure_ascii=False
            ),
            "rewritten_query": obj.get("rewritten_query"),
            "latency_sec": latency,
            "error": error
        })

    holdout_df = pd.DataFrame(holdout_rows)

    holdout_df.to_csv(
        RESULTS_DIR / "week3_prompt_v21_holdout_queries.csv",
        index=False
    )

    print("Holdout queries processed:", len(holdout_df))
else:
    print("Holdout run skipped until development gate is satisfactory.")

## 10. Export V2.1 results

In [ ]:
from datetime import datetime
import shutil

report = f"""
# Amazon Search / LTR Project
## Week 3 Prompt V2.1 — Canonical Attribute Evaluation

### Development Set
50 manually annotated queries.

### Manual Metrics
- Intent Accuracy: {intent_acc:.4f}
- Normalized Attribute Precision: {precision:.4f}
- Normalized Attribute Recall: {recall:.4f}
- Normalized Attribute F1: {attr_f1:.4f}
- Target F1: >= {TARGET_ATTRIBUTE_F1:.2f}

### Retrieval
- Original Recall@50: {original_r50:.4f}
- V2.1 Rewrite Recall@50: {v21_r50:.4f}

### Gate
{'PASS' if gate else 'NOT YET PASSED'}

### Methodological Change
Brand, category, core product/product type, use case, and price are evaluated in their
own structured fields and are not double-counted as attributes. Attribute names are
restricted to a canonical vocabulary before computing pair-level precision/recall/F1.

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}
""".strip()

(RESULTS_DIR / "Amazon_Week3_PromptV21_Report.md").write_text(
    report + "\n",
    encoding="utf-8"
)

(RESULTS_DIR / "prompt_v21.txt").write_text(
    PROMPT_V21 + "\n",
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    "amazon_week3_prompt_v21_results",
    "zip",
    root_dir=RESULTS_DIR
)

print(report)
print()
print("Created:", zip_path)

if in_colab():
    from google.colab import files
    files.download(zip_path)